# Статистические и ML-прогнозы, горизонт и утечки

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Сравнивать простую статистическую модель и ML-прогноз на одном горизонте без утечки.

## Два способа записать прошлое

Авторегрессия прогнозирует следующий ответ как сумму прошлых значений со свободным членом: $\hat y_t=b+\sum_{j=1}^p a_j y_{t-j}$. Это статистическая модель и одновременно линейная регрессия по лаговым признакам. Более сложный ML-метод может использовать те же лаги, календарь и доступные внешние признаки.

Нельзя сравнивать их на разных временных частях или горизонтах. Для прогноза дальше одного шага можно строить отдельные модели горизонтов либо рекурсивно подставлять собственные прогнозы. Во втором случае ошибки накапливаются, а будущие истинные значения подставлять нельзя.

## Разобранный пример: повторение ошибки

Модель следующего шага задаёт y_next=y_last+2. Последнее известное 10. Рекурсивный прогноз двух шагов [12,14]. Если первый истинный ответ оказался 11, это ещё не разрешает заменить 12 на 11 при оценке прогноза, который должен был быть выдан целиком заранее.

Статистические модели могут отдельно описывать тренд, сезонность и шум. До сложных моделей проверьте разности и сезонный baseline. Преобразование ряда тоже должно обращаться только к прошлому.

## Оценка и ограничения

Считайте метрики по каждому горизонту, а не только среднее всех дней. Необычное событие, изменение режима и внешние факторы могут нарушить старую закономерность. Интервалы прогноза обычно шире интервалов для среднего при том же уровне доверия: будущий ответ содержит и шум, и ошибку параметров.

В ноутбуке LinearRegression обучается по одному лаговому признаку на раннем отрезке придуманного ряда. Линию, наивный baseline и знакомое дерево сравниваем на одной поздней части, каждый раз на горизонте один шаг. В `X[10:]` доступны истинные предыдущие значения: это серия прогнозов, выдаваемых последовательно после каждого нового наблюдения, а не один прогноз всего отрезка заранее. После этого отдельный цикл строит два будущих шага рекурсивно от последнего известного значения ряда. В этом тренде линия продолжает рост точно; дерево за пределами обучающих значений остаётся на постоянном ответе. Усложнение метода не гарантирует лучшей экстраполяции — прогноза за границами виденных значений. Идеальный искусственный закон выбран для понятной механики и не обещает столь же точный реальный прогноз.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
series = 2 * np.arange(16, dtype=float) + 1
X, y = series[:-1].reshape(-1, 1), series[1:]
model = LinearRegression().fit(X[:10], y[:10])
predicted = model.predict(X[10:])
assert predicted.shape == y[10:].shape
assert np.allclose(predicted, y[10:])
baseline = X[10:, 0]  # Последнее известное значение для каждого одношагового прогноза.
tree = DecisionTreeRegressor(max_depth=2, random_state=17).fit(X[:10], y[:10])
tree_predicted = tree.predict(X[10:])
linear_mse = ((predicted - y[10:])**2).mean()
baseline_mse = ((baseline - y[10:])**2).mean()
tree_mse = ((tree_predicted - y[10:])**2).mean()
assert np.isclose(baseline_mse, 4.)
assert linear_mse < baseline_mse < tree_mse
print("MSE: линия, baseline, дерево:", linear_mse, baseline_mse, tree_mse)
value = float(series[-1])
future = []
for _ in range(2):
    value = float(model.predict([[value]])[0])
    future.append(value)
assert np.allclose(future, [33., 35.])
print(future)


## Типичные ошибки

Если прогноз выдаётся сразу на весь горизонт, при его проверке нельзя подставлять фактические ответы промежуточных шагов. Сохраняйте даты и частоту.

## Задачи

### 1. Базовая

Что входит в авторегрессию?

<details>
<summary>Подсказка</summary>

Предыдущие значения ряда.

</details>



### 2. Базовая

Можно ли при рекурсивном прогнозе подставлять неизвестный будущий y?

<details>
<summary>Подсказка</summary>

В момент выдачи его ещё нет.

</details>



### 3. Прикладная

Последнее 5, правило +2. Найдите два шага.

<details>
<summary>Подсказка</summary>

Второй использует первый прогноз.

</details>



### 4. Прикладная

Почему полезно считать ошибку отдельно по горизонту?

<details>
<summary>Подсказка</summary>

Дальний ответ обычно труднее.

</details>



### 5. Олимпиадная

Достаточно ли обучить сильный бустинг по лагам, чтобы устранить сдвиг режима?

<details>
<summary>Подсказка</summary>

Связь будущего с прошлым могла измениться.

</details>



### 6. Олимпиадная

Почему интервал среднего не равен интервалу следующего наблюдения?

<details>
<summary>Подсказка</summary>

В будущем есть индивидуальный шум.

</details>



## Итоги и самопроверка

Сравните прямой и рекурсивный прогноз. Какие признаки календаря доступны заранее, а какие внешние значения ещё неизвестны?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Лаги с коэффициентами и, возможно, свободным членом.

### Задача 2

Нет. Используются собственные предсказания или другой заранее заданный способ.

### Задача 3

[7,9].

### Задача 4

Среднее может скрывать резкое ухудшение на последних шагах.

### Задача 5

Нет. Сложность не создаёт отсутствующие сведения о новом режиме.

### Задача 6

Следующий ответ отклоняется от условного среднего даже при точно известных параметрах.